# Supervised PatchTST

An end-to-end supervised classifier built from the repository's standalone PatchTST encoder.

In [ ]:
#| default_exp supervised_patchtst

In [ ]:
#| export
import torch
from torch import nn

from physiojepa.heads import AttentiveClassifier
from physiojepa.patchtst import PatchTFTSimple

In [ ]:
#| export
class TemporalMeanPoolClassifier(nn.Module):
    """Mean-pool contextualized patches per channel before classification."""

    def __init__(self, d_model, c_in, num_classes=1):
        super().__init__()
        self.linear = nn.Linear(d_model * c_in, num_classes)

    def forward(self, x):
        """Pool ``[batch, channels, d_model, patches]`` into logits."""
        x = x.mean(dim=-1)
        return self.linear(x.flatten(start_dim=1))

In [ ]:
#| export
class SupervisedPatchTST(nn.Module):
    """Standalone PatchTST encoder with an end-to-end supervised head."""

    def __init__(
        self,
        c_in,
        patch_size,
        patch_stride,
        num_patches,
        d_model,
        n_heads,
        d_ff,
        num_layers,
        augmentations=None,
        mask_ratio=0.0,
        shared_embedding=False,
        dropout=0.0,
        attn_dropout=0.0,
        act='gelu',
        pre_norm=False,
        pe_type='tAPE',
        qkv_bias=True,
        init_std=0.02,
        tokenizer_type='simple',
        tokenizer_kwargs=None,
        classifier_mlp_ratio=4.0,
        classifier_depth=1,
        classifier_init_std=0.02,
        classifier_qkv_bias=True,
        classifier_complete_block=True,
        classifier_affine=False,
        classifier_type='attentive',
        num_classes=1,
    ):
        super().__init__()
        if augmentations is None:
            augmentations = []
        if tokenizer_kwargs is None:
            tokenizer_kwargs = {}

        # This is the standalone PatchTST comparison encoder. The only
        # architectural change is replacing its reconstruction head with
        # a selectable supervised classification head.
        self.encoder = PatchTFTSimple(
            c_in=c_in,
            patch_size=patch_size,
            patch_stride=patch_stride,
            num_patches=num_patches,
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            num_layers=num_layers,
            augmentations=augmentations,
            mask_ratio=mask_ratio,
            shared_embedding=shared_embedding,
            pretrain_head=False,
            dropout=dropout,
            attn_dropout=attn_dropout,
            act=act,
            pre_norm=pre_norm,
            pe_type=pe_type,
            qkv_bias=qkv_bias,
            init_std=init_std,
            tokenizer_type=tokenizer_type,
            tokenizer_kwargs=tokenizer_kwargs,
        )
        classifier_type = classifier_type.lower()
        if classifier_type == 'attentive':
            self.classifier = AttentiveClassifier(
                embed_dim=d_model,
                num_heads=n_heads,
                mlp_ratio=classifier_mlp_ratio,
                depth=classifier_depth,
                init_std=classifier_init_std,
                qkv_bias=classifier_qkv_bias,
                num_classes=num_classes,
                complete_block=classifier_complete_block,
                affine=classifier_affine,
                c_in=c_in,
            )
        elif classifier_type == 'mean':
            self.classifier = TemporalMeanPoolClassifier(
                d_model=d_model,
                c_in=c_in,
                num_classes=num_classes,
            )
        else:
            raise ValueError(
                "classifier_type must be either 'attentive' or 'mean', "
                f"got {classifier_type!r}"
            )

    def forward(self, x):
        return self.classifier(self.encoder(x))

In [ ]:
# Standard supervised PatchTST: rotary positional encoding with a linear tokenizer and the default attentive classifier.
# This general-purpose configuration checks output shape, encoder/classifier gradients, and head separation.
torch.manual_seed(16)
model = SupervisedPatchTST(
    c_in=3,
    patch_size=4,
    patch_stride=4,
    num_patches=8,
    d_model=16,
    n_heads=4,
    d_ff=32,
    num_layers=2,
    shared_embedding=False,
    pe_type='rotary',
    tokenizer_type='linear',
    num_classes=1,
)
x = torch.randn(2, 3, 32)
target = torch.tensor([[0.0], [1.0]])
logits = model(x)
assert logits.shape == (2, 1)
assert not hasattr(model.encoder, 'head')
nn.BCEWithLogitsLoss()(logits, target).backward()
assert any(p.grad is not None for p in model.encoder.parameters())
assert any(p.grad is not None for p in model.classifier.parameters())

# Compact FCN-comparison model: 1-second patches, simple tokenization, and mean pooling.
# Its width/depth are chosen to match the 260,281-parameter FCN baseline, not to maximize capacity.
compact_model = SupervisedPatchTST(
    c_in=3,
    patch_size=125,
    patch_stride=125,
    num_patches=1800,
    d_model=96,
    n_heads=4,
    d_ff=384,
    num_layers=2,
    shared_embedding=False,
    pe_type='rotary',
    tokenizer_type='simple',
    classifier_type='mean',
    num_classes=1,
)
assert sum(p.numel() for p in compact_model.parameters()) == 260_281
# Minimal mean-pooling variant: a small shape-only check for the same classifier interface.
mean_model = SupervisedPatchTST(
    c_in=3,
    patch_size=4,
    patch_stride=4,
    num_patches=8,
    d_model=16,
    n_heads=4,
    d_ff=32,
    num_layers=1,
    shared_embedding=False,
    pe_type='rotary',
    tokenizer_type='linear',
    classifier_type='mean',
    num_classes=1,
)
mean_logits = mean_model(torch.randn(2, 3, 32))
assert mean_logits.shape == (2, 1)

# Configuration guard: unsupported classifier names must fail early rather than silently selecting another head.
try:
    SupervisedPatchTST(
        c_in=3,
        patch_size=4,
        patch_stride=4,
        num_patches=8,
        d_model=16,
        n_heads=4,
        d_ff=32,
        num_layers=1,
        classifier_type='unknown',
    )
except ValueError as exc:
    assert "classifier_type" in str(exc)
else:
    raise AssertionError("invalid classifier_type should raise ValueError")